# BERT Reranker on inf2.xlarge with Triton Inference Server (SDK 2.31)

Deploys [Alibaba-NLP/gte-multilingual-reranker-base](https://huggingface.co/Alibaba-NLP/gte-multilingual-reranker-base)
(~306M param BERT cross-encoder reranker) on **inf2.xlarge** via NVIDIA Triton Inference
Server on Neuron SDK 2.31, then benchmarks throughput and latency.

Fully self-contained and single-instance: this notebook compiles the model, builds the Triton
image, starts the server, benchmarks it, and cleans up -- all on one inf2.xlarge. No second
machine is needed at any point.

## Why inf2.xlarge is the right size for this model

**inf2.xlarge carries a complete Inferentia2 device**: 1 device, **2 NeuronCores, 32 GB** of
device memory. That is the same accelerator found on every larger inf2 size -- the bigger
sizes add host vCPU and RAM, and (from `inf2.24xlarge` up) more devices. For a single-device
workload like this reranker, the extra host resources go unused.

| | inf2.xlarge |
|---|---|
| Inferentia2 devices | 1 |
| NeuronCores | **2** |
| Device memory | 32 GB |
| vCPU | 4 |
| Host RAM | 16 GB |
| On-demand $/hr (us-east-2) | **$0.7582** |

So this notebook runs **2 Triton model instances, one per NeuronCore** -- the instance count
is set by the accelerator, not by the vCPU count.

**Measured on this instance type, this notebook, SDK 2.31 DLAMI `20260813`:**

| Result | Value |
|---|---|
| Peak throughput | **154.8-156.7 inf/s** (BS=1, high concurrency) |
| **Recommended operating point** | **~153 inf/s at 40-51 ms p50** (BS=1, 8 workers) |
| Neuron device utilization | **94-95% avg** (>= 96% in most configs) |
| Host CPU utilization | **~54% avg** on 4 vCPU |
| Accuracy vs CPU FP32 | cosine **0.999999**, Spearman **1.0000** |

Ranges span **four independent full runs on two instances in two regions** (us-east-2 and
us-west-2): peaks of 156.7 / 156.5 / 156.1 / 154.8 inf/s. That ~1.2% spread is the
run-to-run noise floor -- treat any smaller difference as noise.

**The workload is device-bound, and that is the key finding.** Both NeuronCores run at
96-100% while the 4 vCPU host sits around half idle -- and the benchmark client is running on
those same 4 vCPU, so a production deployment with an off-box client has even more headroom.
A 4 vCPU host does **not** starve the Triton Python backend for this model. Step 6
instruments both numbers so you can confirm this on your own hardware rather than taking it
on faith.

## The one thing you must not skip: swap

Compilation is **host-side** work and peaks at **14.75 GB RSS on a 15 GB host**. The DLAMI
ships with **zero swap**, so on a stock instance `neuronx-cc` is OOM-killed -- and the error
message names the compiler, not memory, which makes it easy to misdiagnose. **Step 0 adds
64 GB of swap automatically.** This is not advisory.

Once swap is present, all five bucket graphs compile on inf2.xlarge without trouble.

**Runtime**: **70-80 min end to end on a fresh instance** (measured 80.5 min: ~38 min Docker
build + ~20 min compiling five buckets + accuracy gate + benchmark), or ~30 min if you reuse
prebuilt artifacts via Step 1-alt.
**Prerequisites**: Deep Learning AMI Neuron (Ubuntu 24.04) 20260813 or later, Docker.

## Measured-best compile configuration

| Setting | Effect | Source |
|---|---|---|
| `attn_implementation="eager"` | **+11.4% throughput** | measured on this model, SDK 2.31 |
| `--model-type transformer` | **+28%** -- do NOT drop it | measured |
| `--auto-cast matmult` | accuracy-safe BF16 speedup | measured, cosine >= 0.999998 |
| `--optlevel 2` omitted | no-op (`-O2` is the default) | measured |
| `inline_weights_to_neff` omitted | already default in torch-neuronx 2.15 | measured (byte-identical NEFF) |

**Structural techniques used** (each has earned its place in a prior deployment):
load + warm every bucket inside `initialize()`, best-fit batch dispatch with pad/trim, bucket
list matched to `preferred_batch_size`, mocked-`pb_utils` standalone test, padding-isolation
and ranking-stability assertions, `KIND_MODEL` with explicit core pinning, staggered instance
init, polling `/v2/models/{name}/ready`, and `shutil.copytree` instead of symlinks.


---
## Step 0: Environment check and **mandatory swap**

The swap step below is **load-bearing on inf2.xlarge**, not advisory.

Measured on this instance type: compiling the BS=16 graph peaked at **14.75 GB of total host
RSS plus 1.69 GB of swap**, on a host with **15 GB of usable RAM and 0 swap by default**. The
compiler is a host-side process and does not care that the device has 32 GB. Without swap it
is OOM-killed, and the failure is misleading — you get
`neuronx-cc ... forcibly killed` or a bare `Compilation error`, with no mention of memory.

So this cell **adds 64 GB of swap if it is missing**, rather than printing a warning.

`mkswap --force` is required — on the DLAMI kernel plain `mkswap` silently no-ops.


In [1]:
import os, subprocess, sys, time, json, shutil

def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True,
                       text=True, **kw)
    return r.stdout.strip(), r.stderr.strip(), r.returncode

print('=== Neuron devices ===')
out, _, _ = sh('neuron-ls')
print(out)

# Confirm inf2.xlarge exposes a FULL Inferentia2 device (2 NeuronCores).
# The instance count in config.pbtxt is driven by this, not by vCPU count.
out, _, _ = sh("neuron-ls 2>/dev/null | grep -oP '^\\| \\d+\\s+\\| \\K\\d+' | head -1")
ncores, _, _ = sh("neuron-ls -j 2>/dev/null | python3 -c \"import json,sys; d=json.load(sys.stdin); print(sum(x['nc_count'] for x in d))\" 2>/dev/null")
print(f'\nNeuronCores visible: {ncores or "see table above"}  (expected 2)')

print('\n=== Host resources ===')
out, _, _ = sh('nproc')
print(f'vCPU     : {out}')
out, _, _ = sh("free -g | awk '/Mem:/ {print $2}'")
print(f'RAM      : {out} GB')
out, _, _ = sh("cat /sys/devices/virtual/dmi/id/product_name 2>/dev/null")

print('\n=== Stack versions ===')
for pkg in ['torch-neuronx', 'neuronx-cc', 'torch', 'transformers']:
    out, _, _ = sh(f'pip show {pkg} 2>/dev/null | head -2')
    print(out if out else f'{pkg}: not installed')

print('\n=== Driver / runtime ===')
out, _, _ = sh("dpkg -l | grep -E 'aws-neuronx-dkms|aws-neuronx-runtime-lib' | awk '{print $2, $3}'")
print(out)

try:
    import tritonclient.http  # noqa
    print('\ntritonclient: present')
except ImportError:
    print('\nInstalling tritonclient...')
    sh(f'{sys.executable} -m pip install -q tritonclient[http]')
    print('done')

# ---------------------------------------------------------------- MANDATORY swap
# Measured: 14.75 GB peak RSS + 1.69 GB swap to compile BS=16 on a 15 GB host.
# This is NOT advisory on inf2.xlarge.
out, _, _ = sh("free -g | awk '/Swap/ {print $2}'")
swap_gb = int(out or 0)
print(f'\n=== Swap: {swap_gb} GB ===')
if swap_gb < 16:
    print('Insufficient swap for neuronx-cc on a 16 GB host. Adding 64 GB...')
    print('(this takes 1-2 min)')
    for cmd in ['sudo dd if=/dev/zero of=/swapfile bs=1M count=65536 status=none',
                'sudo chmod 600 /swapfile',
                'sudo mkswap --force /swapfile',   # --force REQUIRED on the DLAMI kernel
                'sudo swapon /swapfile']:
        o, e, rc = sh(cmd)
        if rc != 0:
            print(f'  FAILED: {cmd}\n  {e}')
            raise SystemExit('could not add swap; compilation will be OOM-killed')
    out, _, _ = sh("free -g | awk '/Swap/ {print $2}'")
    print(f'Swap now: {out} GB')
else:
    print('sufficient - OK')

out, _, _ = sh('free -g')
print('\n' + out)

=== Neuron devices ===
instance-type: inf2.xlarge
instance-id: i-<redacted>
+--------+--------+----------+--------+--------------+----------+------+
| NEURON | NEURON |  NEURON  | NEURON |     PCI      |   CPU    | NUMA |
| DEVICE | CORES  | CORE IDS | MEMORY |     BDF      | AFFINITY | NODE |
+--------+--------+----------+--------+--------------+----------+------+
| 0      | 2      | 0-1      | 32 GB  | 0000:00:1f.0 | 0-3      | -1   |
+--------+--------+----------+--------+--------------+----------+------+



NeuronCores visible: 2  (expected 2)

=== Host resources ===
vCPU     : 4
RAM      : 15 GB

=== Stack versions ===


Name: torch-neuronx
Version: 2.9.0.2.15.32035+de43f57c


Name: neuronx-cc
Version: 2.26.6360.0+6f180f47


Name: torch
Version: 2.9.1


Name: transformers
Version: 4.57.6

=== Driver / runtime ===
aws-neuronx-dkms 2.29.0.0
aws-neuronx-runtime-lib 2.33.10.0-3dcef56f0



tritonclient: present

=== Swap: 0 GB ===
Insufficient swap for neuronx-cc on a 16 GB host. Adding 64 GB...
(this takes 1-2 min)


Swap now: 63 GB

total        used        free      shared  buff/cache   available
Mem:              15           0           0           0          13          14
Swap:             63           0          63


---
## Step 1: Compile the reranker for all batch sizes

Compiles one graph per batch size (1, 2, 4, 8, 16) at seq_len=1024. Per-batch-size
compilation is the single highest-impact Triton technique: a dedicated small-batch graph
avoids padding every request up to BS=16.

> The gain was **not** separately A/B'd for this reranker. For scale, the technique's
> originating project measured **+279% to +360%** at small batch on **DistilBERT-base at
> seq_len=128** -- a different, smaller model. Treat that as motivation, not as a prediction
> for this model.

### Compiling on 16 GB of host RAM

This is the part that genuinely differs on inf2.xlarge. Measured on this instance type:

| | Value |
|---|---|
| Peak total host RSS, BS=16 compile | **14.75 GB** |
| Peak swap used | **1.69 GB** |
| Host RAM available | 15 GB |
| BS=16 compile wall time | **502 s** (8.4 min) |
| Resulting artifact (BS=16) | **710.9 MB** |

The compile **does succeed**, but only because Step 0 added swap: it needed ~16.4 GB of
total commit on a 15 GB machine.

This cell compiles all 5 buckets sequentially and reports peak memory for each, so you can
see the headroom directly rather than trusting the numbers above.

**If you would rather not compile here at all**, skip the next cell (the compile loop) and
use **Step 1-alt** instead, which pulls prebuilt artifacts from S3. The shared-configuration
cell above still has to run -- Step 1-alt needs `MODEL_DIR` and `BATCH_SIZES` from it.


### Shared configuration

These four values are used by every later step, including **Step 1-alt**. This cell is
deliberately separate from the compile cell below so that a **deploy-only instance can skip
compilation entirely** and still have everything it needs.

**Run this cell always.**


In [2]:
# Used by Step 1 (compile), Step 1-alt (S3 reuse), Step 2 (config.pbtxt),
# Step 3 (standalone test) and Step 6 (benchmark). Always run this cell.
MODEL_ID    = 'Alibaba-NLP/gte-multilingual-reranker-base'
SEQ_LENGTH  = 1024
BATCH_SIZES = [1, 2, 4, 8, 16]      # must match preferred_batch_size in config.pbtxt
MODEL_DIR   = os.path.expanduser('~/triton_repo/bert_reranker/1')
DOCKER_IMAGE = 'triton-neuron-bert-reranker:sdk231'
os.makedirs(MODEL_DIR, exist_ok=True)
print(f'model     : {MODEL_ID}')
print(f'seq_len   : {SEQ_LENGTH}')
print(f'buckets   : {BATCH_SIZES}')
print(f'model dir : {MODEL_DIR}')
print(f'image     : {DOCKER_IMAGE}')

model     : Alibaba-NLP/gte-multilingual-reranker-base
seq_len   : 1024
buckets   : [1, 2, 4, 8, 16]
model dir : /home/ubuntu/triton_repo/bert_reranker/1
image     : triton-neuron-bert-reranker:sdk231


In [3]:
# The compile loop runs in a SUBPROCESS so the notebook kernel never claims a
# NeuronCore (see the "Why this runs in a subprocess" note under the accuracy gate).
#
# SKIP THIS CELL if you are pulling prebuilt artifacts -- see Step 1-alt below.
compile_src = r"""
import os, time, sys, threading, subprocess
import torch, torch_neuronx
from transformers import AutoTokenizer, AutoModelForSequenceClassification

os.environ['NEURON_RT_LOG_LEVEL'] = 'ERROR'

MODEL_ID   = 'Alibaba-NLP/gte-multilingual-reranker-base'
SEQ_LENGTH = 1024
BATCH_SIZES = [1, 2, 4, 8, 16]
MODEL_DIR  = os.path.expanduser('~/triton_repo/bert_reranker/1')

# Measured-best flags. See the markdown above for the numbers behind each.
COMPILER_ARGS = ['--model-type', 'transformer',   # +28% on this model
                 '--auto-cast', 'matmult']        # accuracy-safe BF16 matmuls
# --optlevel 2 omitted: -O2 is the default (no-op)
# inline_weights_to_neff omitted: already the default in torch-neuronx 2.15

# --- host memory watchdog -------------------------------------------------
# On a 16 GB host this is the metric that decides whether the compile survives.
_peak = {'rss': 0, 'swap': 0}
_stop = threading.Event()

def _mon():
    while not _stop.is_set():
        try:
            o = subprocess.run("ps -eo rss --no-headers | awk '{s+=$1} END {print s}'",
                               shell=True, capture_output=True, text=True).stdout.strip()
            _peak['rss'] = max(_peak['rss'], int(o or 0))
            s = subprocess.run("free -m | awk '/Swap/ {print $3}'", shell=True,
                               capture_output=True, text=True).stdout.strip()
            _peak['swap'] = max(_peak['swap'], int(s or 0))
        except Exception:
            pass
        time.sleep(2)

threading.Thread(target=_mon, daemon=True).start()

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, torchscript=True, trust_remote_code=True,
    attn_implementation='eager')       # +11.4% -- avoids NewSdpaAttention
model.eval()

attn = type(model.base_model.encoder.layer[0].attention).__name__
print(f'Model: {MODEL_ID}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Attention class: {attn}' + ('  <-- GOOD' if attn == 'NewAttention'
                                    else '  <-- WARNING: expected NewAttention'))
print(f'Compiling {BATCH_SIZES} at seq_len={SEQ_LENGTH}')

for bs in BATCH_SIZES:
    out_path = os.path.join(MODEL_DIR, f'model_bs{bs}.pt')
    if os.path.exists(out_path):
        print(f'  BS={bs}: already compiled, skipping')
        continue
    e = tok(['What is machine learning?'] * bs,
            ['Machine learning is a subset of artificial intelligence.'] * bs,
            return_tensors='pt', max_length=SEQ_LENGTH,
            padding='max_length', truncation=True)
    _peak['rss'] = _peak['swap'] = 0
    t0 = time.time()
    traced = torch_neuronx.trace(model, (e['input_ids'], e['attention_mask']),
                                 compiler_args=COMPILER_ARGS)
    torch.jit.save(traced, out_path)
    print(f'  BS={bs}: {os.path.getsize(out_path)/2**20:.1f} MB in {time.time()-t0:.0f}s'
          f'  [peak host RSS {_peak["rss"]/1024/1024:.2f} GB, '
          f'swap used {_peak["swap"]/1024:.2f} GB]', flush=True)
    del traced

_stop.set()
print('COMPILE DONE')
"""

CSCRIPT = os.path.expanduser('~/triton_repo/compile_all.py')
os.makedirs(os.path.dirname(CSCRIPT), exist_ok=True)
with open(CSCRIPT, 'w') as f:
    f.write(compile_src)

print('compiling in subprocess...')
print('inf2.xlarge: expect ~8-9 min for BS=16 alone, ~25-30 min for all five buckets.')
r = subprocess.run([sys.executable, CSCRIPT], capture_output=True, text=True, timeout=10800)
print(r.stdout[-5000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-3000:])
    if 'killed' in r.stderr.lower() or r.returncode == -9:
        print('\n' + '=' * 72)
        print('Looks like an OOM kill. Confirm swap is active (Step 0) and retry.')
        print('neuronx-cc needs ~16 GB of commit for this model; the host has 15 GB.')
        print('=' * 72)
    raise SystemExit('compilation failed')

print('\nCompiled graphs:')
for f in sorted(os.listdir(MODEL_DIR)):
    if f.endswith('.pt'):
        print(f'  {f}: {os.path.getsize(os.path.join(MODEL_DIR, f))/2**20:.1f} MB')

compiling in subprocess...
inf2.xlarge: expect ~8-9 min for BS=16 alone, ~25-30 min for all five buckets.


........
Compiler status PASS
Model: Alibaba-NLP/gte-multilingual-reranker-base
Parameters: 305,959,681
Attention class: NewAttention  <-- GOOD
Compiling [1, 2, 4, 8, 16] at seq_len=1024
  BS=1: 675.9 MB in 165s  [peak host RSS 10.20 GB, swap used 0.00 GB]
......
Compiler status PASS
  BS=2: 678.4 MB in 130s  [peak host RSS 10.55 GB, swap used 0.00 GB]
........
Compiler status PASS
  BS=4: 682.6 MB in 163s  [peak host RSS 9.21 GB, swap used 0.00 GB]
............
Compiler status PASS
  BS=8: 693.1 MB in 250s  [peak host RSS 11.26 GB, swap used 0.00 GB]
......................
Compiler status PASS
  BS=16: 710.9 MB in 448s  [peak host RSS 14.82 GB, swap used 1.31 GB]
COMPILE DONE


Compiled graphs:
  model_bs1.pt: 675.9 MB
  model_bs16.pt: 710.9 MB
  model_bs2.pt: 678.4 MB
  model_bs4.pt: 682.6 MB
  model_bs8.pt: 693.1 MB


### Step 1-alt (optional): compile on one machine, deploy to many

Skip this for a one-off deployment -- everything above already works on this single
inf2.xlarge. Use it when you are standing up **several** instances, or when you want a
**deploy-only instance that never runs the compiler** and therefore needs no swap.

Both expensive steps -- the ~20-25 min compile and the ~25-40 min Docker build -- are **pure
host-side work**. The compiled graph depends on the *target device*, not on the host that
produced it, and every inf2 size carries the same Inferentia2 device. So build once and reuse.

#### What transfers, and what it must match

| Artifact | Size | Must match on the consumer |
|---|---|---|
| `model_bs{1,2,4,8,16}.pt` | ~3.3 GB total | SDK version, **instance family** (inf2), seq_len, compiler flags |
| `triton-neuron-bert-reranker:sdk231` image | ~6 GB gzipped | nothing device-specific -- it is a plain container |

**Hard constraints.** A graph compiled under one of these will not load, or will
silently misbehave, under another:

- **Same Neuron SDK.** Compile and serve on the same DLAMI (`20260813`, SDK 2.31). A NEFF from
  a different SDK may fail at `nrt_load`.
- **Same instance family.** inf2.xlarge -> inf2.8xlarge is fine (identical device). **inf2 ->
  trn2 is not** -- different architecture and LNC model; recompile.
- **Same `seq_len` and bucket list.** These are baked into the graphs; `config.pbtxt` is
  generated from `BATCH_SIZES` and must agree.

**Verified for this model:** a mixed-provenance repo -- BS=16 compiled locally, BS=1/2/4/8
compiled on a *different* instance and pulled from S3 -- served correctly, passing the accuracy
gate (cosine 0.999999), the bit-exact padding-isolation check, and ranking stability across
every bucket.

> **One caveat worth stating.** This is the `torch_neuronx.trace()` path, compiling on an inf2
> host *for* inf2 -- which is only artifact transfer, not true cross-compilation. Compiling on
> a **non-Neuron** host via `NEURON_PLATFORM_TARGET_OVERRIDE` is a different, riskier thing:
> elsewhere in our testing an XLA-path artifact built that way ran one forward pass correctly
> and then **hung indefinitely on the second**. If you cross-compile on a host with no Neuron
> device, validate with **repeated** forward passes, not a single one.

#### Producer -- run once on any inf2 instance

After Step 1 (compile) and Step 4 (image build) have completed:

```bash
BUCKET=s3://YOUR_BUCKET/bert-reranker

# 1. the five compiled graphs (~3.3 GB)
tar -C ~/triton_repo/bert_reranker/1 -czf /tmp/neffs.tar.gz .
aws s3 cp /tmp/neffs.tar.gz $BUCKET/neffs.tar.gz

# 2. the Triton image (~6 GB gzipped; takes a few minutes)
docker save triton-neuron-bert-reranker:sdk231 | gzip > /tmp/triton-img.tar.gz
aws s3 cp /tmp/triton-img.tar.gz $BUCKET/triton-img.tar.gz
```

Record the SDK version you built on. The consumer must match it.

#### Consumer -- on each additional instance

1. Run **Step 0** (environment check). Swap is **not** required if you will not compile, but it
   is harmless to leave enabled.
2. Run the **shared-configuration cell** above. Step 1-alt needs `MODEL_DIR` and
   `BATCH_SIZES` from it.
3. **Skip the compile cell.**
4. Set `USE_S3_ARTIFACTS = True` and `S3_PREFIX` in the cell below, then run it. It downloads
   the graphs and `docker load`s the image.
5. Continue from the **accuracy gate** onward as normal. Step 4 detects the loaded image and
   skips the build.

Requires an instance role with `s3:GetObject` on the bucket.

The cell is a **no-op unless you opt in**, so the default path stays self-contained.


In [4]:
USE_S3_ARTIFACTS = False          # set True on a consumer instance
S3_PREFIX        = 's3://YOUR_BUCKET/bert-reranker'

if not USE_S3_ARTIFACTS:
    print('skipped (USE_S3_ARTIFACTS = False) - using locally compiled/built artifacts')
else:
    # Fail loudly if the shared-config cell was not run, rather than NameError-ing
    # halfway through a 6 GB download.
    for _v in ('MODEL_DIR', 'BATCH_SIZES', 'DOCKER_IMAGE'):
        if _v not in dir():
            raise SystemExit(f'{_v} is undefined - run the shared-configuration cell '
                             f'(above Step 1) before this one.')

    # ---- compiled graphs ----
    have = sorted(f for f in os.listdir(MODEL_DIR) if f.endswith('.pt'))
    want = [f'model_bs{b}.pt' for b in BATCH_SIZES]
    if all(w in have for w in want):
        print(f'all {len(want)} graphs already present locally, skipping NEFF download')
    else:
        print(f'downloading graphs (~3.3 GB)...')
        o, e, rc = sh(f'aws s3 cp {S3_PREFIX}/neffs.tar.gz /tmp/neffs.tar.gz --no-progress')
        if rc != 0:
            raise SystemExit(f'S3 download failed: {e}\n'
                             f'Check the bucket path and that this instance has s3:GetObject.')
        o, e, rc = sh(f'tar -C {MODEL_DIR} -xzf /tmp/neffs.tar.gz')
        if rc != 0:
            raise SystemExit(f'extract failed: {e}')

        # Verify we got exactly the buckets config.pbtxt will advertise, BEFORE
        # cleaning up the tarball. A missing bucket would otherwise surface much
        # later as a confusing Triton model-load error.
        have = sorted(f for f in os.listdir(MODEL_DIR) if f.endswith('.pt'))
        missing = [w for w in want if w not in have]
        if missing:
            raise SystemExit(f'artifact bundle is incomplete, missing {missing}. '
                             f'The producer\'s BATCH_SIZES must match this '
                             f'notebook\'s {BATCH_SIZES}. Tarball left at '
                             f'/tmp/neffs.tar.gz for inspection.')
        if os.path.exists('/tmp/neffs.tar.gz'):
            os.remove('/tmp/neffs.tar.gz')
        for f in have:
            print(f'  {f}: {os.path.getsize(os.path.join(MODEL_DIR, f))/2**20:.1f} MB')
        print('all buckets present')

    # ---- Triton image ----
    img, _, _ = sh(['docker', 'images', '-q', DOCKER_IMAGE])
    if img:
        print(f'{DOCKER_IMAGE} already present locally, skipping image download')
    else:
        print('downloading Triton image (~6 GB compressed, several minutes)...')
        o, e, rc = sh(f'aws s3 cp {S3_PREFIX}/triton-img.tar.gz /tmp/triton-img.tar.gz '
                      f'--no-progress')
        if rc != 0:
            raise SystemExit(f'S3 download failed: {e}')
        print('loading into docker...')
        o, e, rc = sh('gunzip -c /tmp/triton-img.tar.gz | docker load')
        print(o or e)
        os.remove('/tmp/triton-img.tar.gz')
        img, _, _ = sh(['docker', 'images', '-q', DOCKER_IMAGE])
        if not img:
            raise SystemExit(f'docker load did not produce {DOCKER_IMAGE}')
        print(f'{DOCKER_IMAGE} loaded - Step 4 will skip the build')

skipped (USE_S3_ARTIFACTS = False) - using locally compiled/built artifacts


### Accuracy gate

Validate the compiled graph against a CPU FP32 reference **before** deploying. For a
reranker what matters is whether the *relevant* passages stay on top, not bit-exactness, so
we check cosine similarity, Spearman correlation, and the top-3 set.

**Why this runs in a subprocess.** A Neuron core can be held by only **one process at a
time**, and once a process initializes the Neuron runtime it keeps that claim for its whole
lifetime -- `del model` + `gc.collect()` does **not** give the cores back (verified). If the
notebook kernel ever loads a traced model, the Step 3 test and the Step 5 Triton container
both fail with `NRT_FAILURE in nrt_init()`. So every device-touching step in this notebook
runs in a **subprocess**, and the kernel itself stays off the device.


In [5]:
ACC_SCRIPT = os.path.expanduser('~/triton_repo/accuracy_gate.py')

acc_src = r"""
import os, sys
import numpy as np
import torch, torch_neuronx
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_ID  = 'Alibaba-NLP/gte-multilingual-reranker-base'
SEQ       = 1024
MODEL_DIR = os.path.expanduser('~/triton_repo/bert_reranker/1')

QUERY = 'What are the benefits of renewable energy?'
PASSAGES = [
    'Renewable energy sources like solar and wind power produce electricity without '
    'greenhouse gas emissions, reducing climate change impacts.',
    'The stock market experienced significant volatility in Q4, with tech stocks leading '
    'the decline amid rising interest rates.',
    'Solar panels have become 90% cheaper over the past decade, making renewable energy '
    'cost-competitive with fossil fuels.',
    'The history of ancient Rome spans over a thousand years, from its founding in 753 BC.',
    'Wind energy creates more jobs per megawatt than coal or natural gas power plants.',
    'Python is a popular programming language known for its readability.',
]
RELEVANT = {0, 2, 4}

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
enc = tok([QUERY]*len(PASSAGES), PASSAGES, return_tensors='pt',
          max_length=SEQ, padding='max_length', truncation=True)

# CPU FP32 reference -- eager attention to match what we compiled
cpu = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, trust_remote_code=True, attn_implementation='eager').eval()
with torch.no_grad():
    o = cpu(enc['input_ids'], enc['attention_mask'])
ref = (o[0] if isinstance(o, (tuple, list)) else o.logits)[:, 0].float().numpy()
del cpu

# Neuron BS=16 graph, padded up then sliced back
neuron = torch.jit.load(os.path.join(MODEL_DIR, 'model_bs16.pt'))
pad  = 16 - len(PASSAGES)
ids  = torch.cat([enc['input_ids'],      enc['input_ids'][:1].repeat(pad, 1)])
mask = torch.cat([enc['attention_mask'], enc['attention_mask'][:1].repeat(pad, 1)])
with torch.no_grad():
    o = neuron(ids, mask)
sc = (o[0] if isinstance(o, (tuple, list)) else o.logits)[:len(PASSAGES), 0].float().numpy()

def spearman(a, b):
    ra, rb = np.argsort(np.argsort(-a)), np.argsort(np.argsort(-b))
    return float(np.corrcoef(ra, rb)[0, 1])

cos  = float(np.dot(sc, ref) / (np.linalg.norm(sc) * np.linalg.norm(ref)))
top3 = set(np.argsort(-sc)[:3].tolist())

print(f'cpu fp32 scores : {np.round(ref, 4).tolist()}')
print(f'neuron scores   : {np.round(sc, 4).tolist()}')
print(f'cosine vs cpu fp32 : {cos:.6f}')
print(f'max abs diff       : {np.max(np.abs(sc-ref)):.6f}')
print(f'spearman           : {spearman(sc, ref):.4f}')
print(f'top-3 == relevant  : {top3 == RELEVANT}  (got {sorted(top3)})')

assert cos > 0.999, f'accuracy regression: cosine {cos}'
assert top3 == RELEVANT, f'relevant passages not top-3: {sorted(top3)}'
print('ACCURACY GATE PASSED')
"""

with open(ACC_SCRIPT, 'w') as f:
    f.write(acc_src)

r = subprocess.run([sys.executable, ACC_SCRIPT], capture_output=True, text=True, timeout=1800)
print(r.stdout[-3000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-2500:])
    raise SystemExit('accuracy gate failed - do not deploy')

cpu fp32 scores : [1.2476999759674072, -3.3083999156951904, 0.9027000069618225, -3.327199935913086, 0.9269000291824341, -3.320499897003174]
neuron scores   : [1.2446000576019287, -3.3076000213623047, 0.9004999995231628, -3.326200008392334, 0.9203000068664551, -3.3203999996185303]
cosine vs cpu fp32 : 0.999999
max abs diff       : 0.006607
spearman           : 1.0000
top-3 == relevant  : True  (got [0, 2, 4])
ACCURACY GATE PASSED



---
## Step 2: Write the Triton model repository

Two files: `config.pbtxt` and `model.py`.

### Techniques ported into `model.py`

| # | Technique | Why |
|---|---|---|
| 1 | **Compile/load + warm every batch size in `initialize()`** | All graphs are hot before the first client request. A customer benchmark that skipped this saw cold-start jitter; it was the one material difference found in a prior review. |
| 2 | **Best-fit dispatch** — smallest graph that fits, then pad/trim | Avoids padding a BS=1 request to BS=16. (+279-360% at small batch when measured on DistilBERT-base/seq=128; not re-measured here.) |
| 3 | **Bucket list == `preferred_batch_size`** | Keeps the dynamic batcher from forming batches no graph serves well. |
| 4 | **Works without `tritonserver`** (`pb_utils` import guarded) | Enables the standalone test in Step 3. |
| 5 | **Staggered init** (`STAGGER_SEC * instance_idx`) | Avoids Neuron runtime races when several instances load at once. |
| 6 | **Explicit core pinning** via `NEURON_RT_VISIBLE_CORES` | One instance per NeuronCore, deterministically assigned. |


In [6]:
REPO = os.path.expanduser('~/triton_repo/bert_reranker')

# ---------------------------------------------------------------- config.pbtxt
# inf2.xlarge has 2 NeuronCores -> 2 instances, one per core.
# The instance count is set by the ACCELERATOR, not by the vCPU count. Do not
# reduce it to "match" the 4 vCPU host; that would idle half the device.
NUM_INSTANCES = 2
config = f"""name: "bert_reranker"
backend: "python"
max_batch_size: 16

input [
  {{ name: "input_ids"      data_type: TYPE_INT64 dims: [ {SEQ_LENGTH} ] }},
  {{ name: "attention_mask" data_type: TYPE_INT64 dims: [ {SEQ_LENGTH} ] }}
]

output [
  {{ name: "score" data_type: TYPE_FP32 dims: [ 1 ] }}
]

instance_group [
  {{ kind: KIND_MODEL count: {NUM_INSTANCES} }}
]

# preferred_batch_size MUST match BATCH_SIZES compiled in Step 1, so every batch the
# dynamic batcher forms has an exactly-matching graph.
dynamic_batching {{
  preferred_batch_size: [ {', '.join(str(b) for b in BATCH_SIZES)} ]
  max_queue_delay_microseconds: 5000
}}

parameters [
  {{ key: "SEQ_LENGTH"    value: {{ string_value: "{SEQ_LENGTH}" }} }},
  {{ key: "BATCH_SIZES"   value: {{ string_value: "{','.join(str(b) for b in BATCH_SIZES)}" }} }},
  {{ key: "NUM_INSTANCES" value: {{ string_value: "{NUM_INSTANCES}" }} }},
  {{ key: "STAGGER_SEC"   value: {{ string_value: "5" }} }}
]
"""
with open(os.path.join(REPO, 'config.pbtxt'), 'w') as f:
    f.write(config)
print('wrote config.pbtxt')
print(config)

wrote config.pbtxt
name: "bert_reranker"
backend: "python"
max_batch_size: 16

input [
  { name: "input_ids"      data_type: TYPE_INT64 dims: [ 1024 ] },
  { name: "attention_mask" data_type: TYPE_INT64 dims: [ 1024 ] }
]

output [
  { name: "score" data_type: TYPE_FP32 dims: [ 1 ] }
]

instance_group [
  { kind: KIND_MODEL count: 2 }
]

# preferred_batch_size MUST match BATCH_SIZES compiled in Step 1, so every batch the
# dynamic batcher forms has an exactly-matching graph.
dynamic_batching {
  preferred_batch_size: [ 1, 2, 4, 8, 16 ]
  max_queue_delay_microseconds: 5000
}

parameters [
  { key: "SEQ_LENGTH"    value: { string_value: "1024" } },
  { key: "BATCH_SIZES"   value: { string_value: "1,2,4,8,16" } },
  { key: "NUM_INSTANCES" value: { string_value: "2" } },
  { key: "STAGGER_SEC"   value: { string_value: "5" } }
]



In [7]:
model_py = r"""
import json
import os
import sys
import time

import numpy as np

try:
    import triton_python_backend_utils as pb_utils
except ImportError:          # standalone-test path (Step 3)
    pb_utils = None


class TritonPythonModel:
    def initialize(self, args):
        import torch
        # torch_neuronx MUST be imported before torch.jit.load of a traced Neuron
        # model -- it registers the __torch__.torch.classes.neuron.Model custom
        # TorchScript class. Without it, load fails with "Unknown type name".
        import torch_neuronx  # noqa: F401

        cfg    = json.loads(args['model_config'])
        params = cfg.get('parameters', {})

        def p(key, default):
            return params.get(key, {}).get('string_value', default)

        self.seq_len  = int(p('SEQ_LENGTH', '1024'))
        self.buckets  = sorted(int(b) for b in p('BATCH_SIZES', '1,2,4,8,16').split(','))
        n_inst        = int(p('NUM_INSTANCES', '2'))
        stagger       = float(p('STAGGER_SEC', '5'))

        idx = int(args.get('model_instance_name', 'x_0').rsplit('_', 1)[-1] or 0)
        core = idx % n_inst

        # Pin this instance to exactly one NeuronCore. Must be set BEFORE the first
        # torch-neuronx call in this process.
        os.environ['NEURON_RT_VISIBLE_CORES'] = str(core)
        self._log(f'instance {idx} -> NeuronCore {core}')

        # Staggered init: avoids a Neuron runtime race when instances load simultaneously.
        if idx > 0 and stagger > 0:
            time.sleep(stagger * idx)

        model_dir = os.path.join(args['model_repository'], args['model_version'])

        # Load one graph per batch size, then WARM each one. Warming here (not in the
        # client) means no request ever pays a cold-graph penalty.
        self.models = {}
        for bs in self.buckets:
            path = os.path.join(model_dir, f'model_bs{bs}.pt')
            if not os.path.exists(path):
                self._log(f'missing {path}, skipping BS={bs}')
                continue
            m = torch.jit.load(path)
            ids  = torch.zeros((bs, self.seq_len), dtype=torch.long)
            mask = torch.ones((bs, self.seq_len), dtype=torch.long)
            with torch.no_grad():
                for _ in range(3):
                    m(ids, mask)
            self.models[bs] = m
            self._log(f'loaded + warmed BS={bs}')

        if not self.models:
            raise RuntimeError('no compiled graphs found')
        self.buckets = sorted(self.models)
        self._log(f'ready; buckets={self.buckets}')

    def _log(self, msg):
        if pb_utils is not None:
            pb_utils.Logger.log_info(f'[bert_reranker] {msg}')
        else:
            print(f'[bert_reranker] {msg}', flush=True)

    def _route(self, n):
        # Smallest compiled graph that fits n. Falls back to the largest.
        for b in self.buckets:
            if b >= n:
                return b
        return self.buckets[-1]

    def execute(self, requests):
        import torch
        responses = []
        for req in requests:
            ids_np  = pb_utils.get_input_tensor_by_name(req, 'input_ids').as_numpy()
            mask_np = pb_utils.get_input_tensor_by_name(req, 'attention_mask').as_numpy()

            n = ids_np.shape[0]
            bs = self._route(n)

            ids  = torch.as_tensor(ids_np,  dtype=torch.long)
            mask = torch.as_tensor(mask_np, dtype=torch.long)

            # Pad up to the graph's batch size; padded rows are sliced off after.
            if n < bs:
                pad = bs - n
                ids  = torch.cat([ids,  ids[:1].repeat(pad, 1)])
                mask = torch.cat([mask, mask[:1].repeat(pad, 1)])
            elif n > bs:
                ids, mask = ids[:bs], mask[:bs]

            with torch.no_grad():
                out = self.models[bs](ids, mask)
            logits = out[0] if isinstance(out, (tuple, list)) else out.logits

            # One score per input row: the CLS logit. Slice back to the real n.
            scores = logits[:n, 0:1].float().numpy().astype(np.float32)
            responses.append(pb_utils.InferenceResponse(
                output_tensors=[pb_utils.Tensor('score', scores)]))
        return responses

    def finalize(self):
        self._log('finalize')
"""

with open(os.path.join(REPO, '1', 'model.py'), 'w') as f:
    f.write(model_py)
print('wrote model.py')

# Show the repo layout Triton will see
for root, dirs, files in os.walk(os.path.expanduser('~/triton_repo')):
    depth = root.replace(os.path.expanduser('~/triton_repo'), '').count(os.sep)
    print('  ' * depth + os.path.basename(root) + '/')
    for fn in sorted(files):
        print('  ' * (depth+1) + fn)

wrote model.py
triton_repo/
  accuracy_gate.py
  compile_all.py
  bert_reranker/
    config.pbtxt
    1/
      model.py
      model_bs1.pt
      model_bs16.pt
      model_bs2.pt
      model_bs4.pt
      model_bs8.pt


---
## Step 3: Standalone backend test (no Triton required)

Ported from the reference project: mock `triton_python_backend_utils` so `model.py`'s full
`initialize()` / `execute()` lifecycle runs in-process. This catches padding, slicing,
routing, and dtype errors in **seconds**, instead of after a 20-minute Docker build.

It also runs a **batch-consistency assertion** — the score for a given (query, passage)
pair must be stable regardless of which batch bucket served it. A per-request test cannot
catch a pad/slice indexing error; this can.


In [8]:
import types

test_src = r"""
import os, sys, types, json
import numpy as np

class _T:
    def __init__(self, name, arr): self.name, self._a = name, arr
    def as_numpy(self): return self._a

class _Resp:
    def __init__(self, output_tensors): self.outputs = output_tensors

mock = types.SimpleNamespace(
    Logger=types.SimpleNamespace(log_info=lambda m: print(m, flush=True)),
    Tensor=_T,
    InferenceResponse=_Resp,
    get_input_tensor_by_name=lambda req, name: req[name],
)
sys.modules['triton_python_backend_utils'] = mock

REPO = os.path.expanduser('~/triton_repo/bert_reranker')
sys.path.insert(0, os.path.join(REPO, '1'))
from model import TritonPythonModel

cfg = open(os.path.join(REPO, 'config.pbtxt')).read()
# minimal parameters dict mirroring config.pbtxt
model_config = json.dumps({'parameters': {
    'SEQ_LENGTH':    {'string_value': '1024'},
    'BATCH_SIZES':   {'string_value': '1,2,4,8,16'},
    'NUM_INSTANCES': {'string_value': '1'},
    'STAGGER_SEC':   {'string_value': '0'},
}})

m = TritonPythonModel()
m.initialize({'model_config': model_config,
              'model_repository': REPO,
              'model_version': '1',
              'model_instance_name': 'bert_reranker_0'})

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('Alibaba-NLP/gte-multilingual-reranker-base',
                                    trust_remote_code=True)
Q = 'What are the benefits of renewable energy?'
P = ['Solar and wind power produce electricity without greenhouse gas emissions.',
     'The stock market experienced volatility in Q4 amid rising rates.',
     'Solar panels are 90% cheaper than a decade ago.']

def infer(n_rows):
    e = tok([Q]*n_rows, (P*8)[:n_rows], return_tensors='np', max_length=1024,
            padding='max_length', truncation=True)
    req = {'input_ids': _T('input_ids', e['input_ids'].astype(np.int64)),
           'attention_mask': _T('attention_mask', e['attention_mask'].astype(np.int64))}
    return m.execute([req])[0].outputs[0]._a

# 1) every routed bucket returns exactly n rows
print('\n--- shape/routing ---')
for n in [1, 2, 3, 5, 8, 12, 16]:
    out = infer(n)
    assert out.shape == (n, 1), f'n={n} returned {out.shape}'
    print(f'  n={n:>2} -> bucket {m._route(n):>2}, shape {out.shape}  OK')

# 2) batch consistency, two separate properties:
#
#    (a) PADDING ISOLATION (must be exact). Row 0's score must not depend on what the
#        other rows of the batch contain. A violation here is a real indexing/masking
#        bug. Tolerance: bit-exact.
#
#    (b) CROSS-GRAPH AGREEMENT (approximate). Each batch size is a separately compiled
#        graph, and with --auto-cast matmult the BF16 accumulation order differs between
#        them, so the same input can score slightly differently on the BS=1 vs BS=8
#        graph. Measured ~2.1e-2 on this model. That is numerics, not a bug, so the
#        tolerance is loose and what we really assert is that RANKING is unaffected.
print('\n--- batch consistency (a): padding isolation, must be exact ---')
FILLER = 'Completely unrelated text about ancient Roman architecture and history.'
for n in [2, 4, 8, 16]:
    e1 = tok([Q]*n, [P[0]]*n, return_tensors='np', max_length=1024,
             padding='max_length', truncation=True)
    e2 = tok([Q]*n, [P[0]] + [FILLER]*(n-1), return_tensors='np', max_length=1024,
             padding='max_length', truncation=True)
    r1 = {'input_ids': _T('input_ids', e1['input_ids'].astype(np.int64)),
          'attention_mask': _T('attention_mask', e1['attention_mask'].astype(np.int64))}
    r2 = {'input_ids': _T('input_ids', e2['input_ids'].astype(np.int64)),
          'attention_mask': _T('attention_mask', e2['attention_mask'].astype(np.int64))}
    s1 = float(m.execute([r1])[0].outputs[0]._a[0, 0])
    s2 = float(m.execute([r2])[0].outputs[0]._a[0, 0])
    print(f'  n={n:>2}: row0 with uniform rows {s1:+.6f} vs varied rows {s2:+.6f}'
          f'  delta {abs(s1-s2):.2e}')
    assert abs(s1 - s2) == 0.0, (
        f'PADDING LEAK at n={n}: row 0 changed by {abs(s1-s2):.3e} when other rows '
        f'changed. This is a real indexing/masking bug.')
print('  row 0 is bit-identical regardless of neighbours -> no padding leak')

print('\n--- batch consistency (b): cross-graph agreement (BF16 numerics) ---')
base = infer(1)[0, 0]
deltas = {}
for n in [2, 4, 8, 16]:
    got = infer(n)[0, 0]
    delta = abs(float(got) - float(base))
    deltas[n] = delta
    print(f'  row0 on BS=1 graph vs BS={m._route(n):>2} graph: '
          f'{base:+.6f} vs {got:+.6f}  delta {delta:.2e}')
    # Loose: BF16 accumulation differs per compiled graph. Tight enough to catch a
    # genuine breakage (which would be order-of-magnitude, not 1e-2).
    assert delta < 0.1, f'cross-graph disagreement too large at n={n}: {delta}'
print(f'  max cross-graph delta {max(deltas.values()):.2e} - BF16 accumulation, expected')

# 3) ranking sanity
print('\n--- ranking ---')
sc = infer(3)[:, 0]
print(f'  scores {np.round(sc,4).tolist()}')
assert sc[0] > sc[1] and sc[2] > sc[1], 'irrelevant passage not ranked last'
print('  relevant passages outrank the irrelevant one  OK')

# 4) ranking stability across graphs -- the property that actually matters. The BF16
#    cross-graph score deltas above are only acceptable if they never reorder results.
print('\n--- ranking stability across batch graphs ---')
mixed = [P[0], P[1], P[2]]          # relevant, irrelevant, relevant
ref_order = None
for n in [4, 8, 16]:
    rows = mixed + [FILLER] * (n - len(mixed))
    e = tok([Q]*n, rows, return_tensors='np', max_length=1024,
            padding='max_length', truncation=True)
    req = {'input_ids': _T('input_ids', e['input_ids'].astype(np.int64)),
           'attention_mask': _T('attention_mask', e['attention_mask'].astype(np.int64))}
    s = m.execute([req])[0].outputs[0]._a[:len(mixed), 0]
    order = tuple(np.argsort(-s).tolist())
    print(f'  bucket {m._route(n):>2}: scores {np.round(s,4).tolist()} -> order {order}')
    if ref_order is None:
        ref_order = order
    assert order == ref_order, (
        f'RANKING CHANGED between graphs: {ref_order} vs {order}. BF16 numerics are '
        f'reordering results - this would be user-visible.')
print(f'  ranking identical across all buckets: {ref_order}')

m.finalize()
print('\nSTANDALONE TEST PASSED')
"""

test_path = os.path.expanduser('~/triton_repo/standalone_test.py')
with open(test_path, 'w') as f:
    f.write(test_src)

# A Neuron device is SINGLE-TENANT -- only one process can hold a core at a time.
# `del` + gc.collect() in the notebook kernel is NOT sufficient to release the runtime's
# claim, so we restart the kernel's device state by running this test in a subprocess
# with the notebook's own Neuron handles already dropped. If the notebook kernel has ever
# loaded a traced model, the subprocess cannot init the runtime.
#
# Belt and braces: ask the kernel to drop everything, then verify no other process holds
# the cores before launching the subprocess.
import gc
for _name in ['neuron', 'traced', 'model']:
    if _name in dir():
        try:
            exec(f'del {_name}')
        except Exception:
            pass
gc.collect()

out, _, _ = sh('neuron-ls -j 2>/dev/null | head -40')
busy, _, _ = sh("ps -eo pid,comm,args | grep -v grep | grep -c 'neuron' || true")

r = subprocess.run([sys.executable, test_path], capture_output=True, text=True, timeout=1800)
print(r.stdout[-5000:])
if r.returncode != 0:
    err = r.stderr[-4000:]
    print('STDERR:', err)
    if 'nrt_init' in err or 'could not be initialized' in err:
        print('\n' + '='*72)
        print('The Neuron runtime could not initialize in the subprocess.')
        print('A Neuron core can be held by only ONE process at a time, and this')
        print('notebook kernel is still holding it from an earlier cell.')
        print('FIX: restart the Jupyter kernel, then run Step 0 -> Step 2 -> Step 3,')
        print('     skipping the Step 1 compile cells (the .pt files are already on disk).')
        print('='*72)
    raise SystemExit('standalone test failed - see message above')

[bert_reranker] instance 0 -> NeuronCore 0
[bert_reranker] loaded + warmed BS=1
[bert_reranker] loaded + warmed BS=2
[bert_reranker] loaded + warmed BS=4
[bert_reranker] loaded + warmed BS=8
[bert_reranker] loaded + warmed BS=16
[bert_reranker] ready; buckets=[1, 2, 4, 8, 16]

--- shape/routing ---
  n= 1 -> bucket  1, shape (1, 1)  OK
  n= 2 -> bucket  2, shape (2, 1)  OK
  n= 3 -> bucket  4, shape (3, 1)  OK
  n= 5 -> bucket  8, shape (5, 1)  OK
  n= 8 -> bucket  8, shape (8, 1)  OK
  n=12 -> bucket 16, shape (12, 1)  OK
  n=16 -> bucket 16, shape (16, 1)  OK

--- batch consistency (a): padding isolation, must be exact ---
  n= 2: row0 with uniform rows +1.304114 vs varied rows +1.304114  delta 0.00e+00
  n= 4: row0 with uniform rows +1.304114 vs varied rows +1.304114  delta 0.00e+00
  n= 8: row0 with uniform rows +1.282856 vs varied rows +1.282856  delta 0.00e+00
  n=16: row0 with uniform rows +1.282856 vs varied rows +1.282856  delta 0.00e+00
  row 0 is bit-identical regardless of 

---
## Step 4: Build the Triton + Neuron Docker image

The stock NVIDIA Triton container has **no torch and no Neuron runtime**, so Triton is built
from source (`r26.01`, python backend only) on top of the Neuron PyTorch inference image.

### Build time on 4 vCPU

The build is overwhelmingly a C++ compile (gRPC, protobuf, Triton core). Measured for this
exact Dockerfile on inf2.xlarge, SDK 2.31 base image, across two separate instances in two
regions: **26.5 min and 38.4 min**. Expect roughly **25-40 min**; the spread is network and
EBS variability on the clone/download phases, not compute.

That is a reasonable one-time cost, and less of a penalty than 4 vCPU might suggest -- large
parts of the build (git clones, downloads, single-threaded CMake configure) are not
CPU-parallel. **Building locally on inf2.xlarge is the recommended path.**

If you are provisioning several instances, Step 1-alt lets you `docker load` an image built
once and stored in S3.

Skipped entirely if the image already exists locally.


In [9]:
# DOCKER_IMAGE comes from the shared-configuration cell above.
out, _, _ = sh(['docker', 'images', '-q', DOCKER_IMAGE])
if out:
    print(f'image already built: {out}')
else:
    dockerfile = r"""ARG BASE_IMAGE=public.ecr.aws/neuron/pytorch-inference-neuronx:2.9.0-neuronx-py312-sdk2.31.0-ubuntu24.04
FROM $BASE_IMAGE

ENV DEBIAN_FRONTEND=noninteractive \
    PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    PJRT_DEVICE=NEURON \
    LD_LIBRARY_PATH="/opt/conda/lib:/opt/aws/neuron/lib:/lib/x86_64-linux-gnu:${LD_LIBRARY_PATH}" \
    PATH="/opt/program:/opt/aws/neuron/bin:/opt/tritonserver/bin:${PATH}"

RUN apt-get update && apt-get install -y --no-install-recommends \
    wget gnupg2 build-essential git nginx pkg-config unzip \
    libssl-dev libcurl4-openssl-dev libgoogle-perftools-dev \
    libnuma-dev libarchive-dev libxml2-dev zlib1g-dev \
    autoconf automake libtool gperf scons patchelf \
    libre2-dev libb64-dev rapidjson-dev libboost-dev \
    cmake cmake-data \
    && rm -rf /var/lib/apt/lists/*

RUN pip3 install --no-cache-dir --upgrade pip setuptools wheel virtualenv build cmake==3.31.10
RUN pip3 install --no-cache-dir "transformers==4.57.6"

RUN git clone --depth=1 --branch=r26.01 https://github.com/triton-inference-server/server.git /server && \
    cd /server && \
    Python3_ROOT_DIR=/opt/conda \
    Python3_EXECUTABLE=/opt/conda/bin/python3 \
    Python3_INCLUDE_DIR=/opt/conda/include/python3.12 \
    Python3_LIBRARY=/opt/conda/lib/libpython3.12.so \
    ./build.py -v --no-container-build --build-dir=/server/build --backend=python \
    --enable-metrics --enable-logging --enable-stats --endpoint="http" --endpoint="grpc" && \
    cp -r /server/build/opt/* /opt/ && \
    cd / && rm -rf /server

EXPOSE 8000 8001 8002
CMD ["tritonserver", "--model-repository=/models"]
"""
    dpath = os.path.expanduser('~/Dockerfile.triton-neuron-sdk231')
    with open(dpath, 'w') as f:
        f.write(dockerfile)
    print('building Triton from source...')
    print('measured: 26-38 min on inf2.xlarge (4 vCPU), across two instances')
    t0 = time.time()
    r = subprocess.run(['docker', 'build', '-f', dpath, '-t', DOCKER_IMAGE, '.'],
                       cwd=os.path.expanduser('~'), capture_output=True,
                       text=True, timeout=3600)
    if r.returncode == 0:
        print(f'built in {(time.time()-t0)/60:.1f} min')
    else:
        print(r.stdout[-2000:]); print(r.stderr[-3000:])
        raise SystemExit('docker build failed')

out, _, _ = sh(['docker', 'images', DOCKER_IMAGE])
print(out)

building Triton from source...
measured: 26-38 min on inf2.xlarge (4 vCPU), across two instances


built in 29.1 min
IMAGE                                ID             DISK USAGE   CONTENT SIZE   EXTRA
triton-neuron-bert-reranker:sdk231   ec0a74dd0232       24.7GB         8.15GB


---
## Step 5: Start the Triton server

Gotchas baked in (each has bitten a prior project):

- Poll **`/v2/models/{name}/ready`**, not `/v2/models/{name}` — the latter can report before
  instances are actually ready.
- Use **`shutil.copytree`**, not symlinks — symlinks break across Docker volume mounts.
- Pass Neuron env vars with **`-e`** — `/etc/environment` is not inherited by containers.
- `--shm-size=4g` for headroom with multiple Python-backend instances.


In [10]:
import requests

CONTAINER = 'triton-bert-reranker'
sh(['docker', 'rm', '-f', CONTAINER])

# copytree, NOT symlink -- symlinks do not resolve inside the container
SERVE_REPO = os.path.expanduser('~/triton_serve_repo')
if os.path.exists(SERVE_REPO):
    shutil.rmtree(SERVE_REPO)
shutil.copytree(os.path.expanduser('~/triton_repo'), SERVE_REPO,
                ignore=shutil.ignore_patterns('standalone_test.py'))

cmd = [
    'docker', 'run', '-d', '--name', CONTAINER,
    '--device', '/dev/neuron0',
    '--shm-size=4g',
    '-p', '8000:8000', '-p', '8001:8001', '-p', '8002:8002',
    '-v', f'{SERVE_REPO}:/models',
    '-e', 'NEURON_RT_LOG_LEVEL=ERROR',
    DOCKER_IMAGE,
    'tritonserver', '--model-repository=/models',
    '--log-verbose=0', '--exit-on-error=true',
]
out, err, rc = sh(cmd)
print('container:', out or err)

print('\nwaiting for readiness (model load + per-bucket warmup takes a few minutes)...')
ready = False
for i in range(120):
    try:
        # /ready, not the bare model endpoint
        if requests.get('http://localhost:8000/v2/health/ready', timeout=2).status_code == 200:
            r2 = requests.get('http://localhost:8000/v2/models/bert_reranker/ready', timeout=2)
            if r2.status_code == 200:
                ready = True
                break
    except Exception:
        pass
    time.sleep(10)
    if i % 6 == 5:
        print(f'  {(i+1)*10}s...')

if not ready:
    out, _, _ = sh(['docker', 'logs', '--tail', '60', CONTAINER])
    print(out)
    raise SystemExit('server did not become ready')

print('SERVER READY')
out, _, _ = sh(f"docker logs {CONTAINER} 2>&1 | grep -c 'ready; buckets'")
print(f'instances reporting ready: {out}')
out, _, _ = sh(f"docker logs {CONTAINER} 2>&1 | grep -E 'NeuronCore|ready; buckets'")
print(out)

container: 6c5048134dc5f9b00ce5162284824f0a1954858d787f509b9c765892484234d7

waiting for readiness (model load + per-bucket warmup takes a few minutes)...


  60s...


  120s...


  180s...


SERVER READY


instances reporting ready: 2
I1001 20:00:52.933268 8 model.py:72] "[bert_reranker] instance 1 -> NeuronCore 1"
I1001 20:00:52.933268 8 model.py:72] "[bert_reranker] instance 0 -> NeuronCore 0"
I1001 20:02:49.445819 8 model.py:72] "[bert_reranker] ready; buckets=[1, 2, 4, 8, 16]"
I1001 20:02:49.657454 8 model.py:72] "[bert_reranker] ready; buckets=[1, 2, 4, 8, 16]"


---
## Step 6: Benchmark, with host-CPU instrumentation

Threaded workers, 10 s per configuration, 5-request warmup, reporting throughput and
P50/P95/P99.

Matrix: concurrency {1, 8, 16, 32} x batch {1, 2, 4, 8, 16}.

### The extra measurement that matters here

There are **4 vCPU** here, and three things compete for them:

- 2 Triton Python-backend instances (each tokenizes nothing at request time, but does
  numpy/torch tensor marshalling per request)
- the HTTP server itself
- **the benchmark client, running on the same host**

So every row below also records **host CPU utilization** and **Neuron device utilization**.
Reading them together is what separates the two possible stories:

| Host CPU | Device util | Interpretation |
|---|---|---|
| ~100% | < 80% | **host-CPU-bound** — the 4 vCPU is the limit, not the accelerator |
| < 80% | ~100% | **device-bound — the 4 vCPU host is not the limit** (this is what we measure) |
| ~100% | ~100% | both saturated |

Note the client is co-resident, so some of the host CPU is measurement overhead rather than
serving cost. A production deployment with an off-box client would see less host pressure
than the numbers below. That caveat is recorded in the results rather than corrected for,
since correcting for it would require guessing.

> **Gotcha worth knowing: `neuron-monitor` must run inside the container.** The Neuron
> runtime lives in the Triton container, so running `neuron-monitor` on the *host* returns
> `neuron_runtime_data: []` -- it reports the hardware inventory correctly but sees no
> runtime, and device utilization silently comes back empty. The cell below therefore
> `docker exec`s into the container to sample. Also note that on SDK 2.31 `neuron-monitor`
> takes a `-c <config.json>`; there is **no** `--sample-interval` or `--count` flag, and
> passing one makes it exit 0 with `unknown flag` on stderr -- a silent no-op if you are not
> checking.
>
> **Two more traps in reading that JSON, both of which silently produce 0%:**
> (1) **the first sample is always 0** -- there is no prior interval to diff against, so
> reading only the first line reports 0% utilization while both cores are pegged;
> (2) **each Triton instance is its own Neuron runtime owning one core**, and every runtime
> reports `0` for cores it does not own, so averaging all the numbers together halves the
> true figure. Take the max per core across runtimes, then average over cores. Both
> behaviours were confirmed directly by sampling under sustained load.


In [11]:
import threading
from queue import Queue
import numpy as np
import tritonclient.http as httpclient

TRITON_URL = 'localhost:8000'
MODEL_NAME = 'bert_reranker'
DURATION   = 10.0

# The tokenizer is CPU-only, so it is safe to use in the notebook kernel (unlike the
# model, which would claim a NeuronCore -- see the Step 1b note).
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

NUM_VCPU = int(subprocess.run('nproc', shell=True, capture_output=True,
                              text=True).stdout.strip() or 4)
print(f'host vCPU: {NUM_VCPU}')

# Install a neuron-monitor config INSIDE the container so device utilization can be
# sampled from where the Neuron runtime actually lives (see HostMon._run).
_nm_cfg = ('{"period":"1s","neuron_runtimes":[{"tag_filter":".*",'
           '"metrics":[{"type":"neuroncore_counters"}]}]}')
_o, _e, _rc = sh(['docker', 'exec', CONTAINER, 'bash', '-lc',
                  f"printf '%s' '{_nm_cfg}' > /tmp/nm.json && echo ok"])
print(f'in-container neuron-monitor config: {_o or _e}')
_probe, _e, _ = sh(['docker', 'exec', CONTAINER, 'bash', '-lc',
                    'timeout 6 /opt/aws/neuron/bin/neuron-monitor -c /tmp/nm.json '
                    '| head -1 | python3 -c "import sys,json; '
                    'print(len(json.loads(sys.stdin.readline())'
                    '.get(chr(110)+chr(101)+chr(117)+chr(114)+chr(111)+chr(110)+'
                    '\'_runtime_data\', [])))"'])
print(f'neuron runtimes visible in container: {_probe or "probe failed"}')


class HostMon:
    """Samples host CPU and Neuron device utilization during a benchmark run.

    Host CPU comes from /proc/stat deltas (no psutil dependency). Device
    utilization comes from `neuron-monitor`, which is sampled best-effort --
    if it is unavailable the benchmark still runs and device util reports None.
    """

    def __init__(self):
        self.cpu, self.dev = [], []
        self._stop = threading.Event()
        self._t = None

    @staticmethod
    def _stat():
        with open('/proc/stat') as f:
            p = f.readline().split()[1:]
        v = [int(x) for x in p[:8]]
        idle = v[3] + v[4]
        return sum(v), idle

    def _run(self):
        tot0, idle0 = self._stat()
        while not self._stop.is_set():
            time.sleep(1.0)
            tot1, idle1 = self._stat()
            dt, di = tot1 - tot0, idle1 - idle0
            if dt > 0:
                self.cpu.append(100.0 * (dt - di) / dt)
            tot0, idle0 = tot1, idle1
            # Device utilization must be sampled INSIDE the Triton container.
            # The Neuron runtime lives in the container, so a host-side
            # neuron-monitor returns an empty neuron_runtime_data list -- it sees
            # the hardware but not the runtime that is driving it. Also note
            # neuron-monitor on SDK 2.31 takes a -c config file; there is no
            # --sample-interval/--count flag.
            try:
                r = subprocess.run(
                    ['docker', 'exec', CONTAINER, 'bash', '-lc',
                     'timeout 6 /opt/aws/neuron/bin/neuron-monitor -c /tmp/nm.json'],
                    capture_output=True, text=True, timeout=10)
                lines = [x for x in r.stdout.strip().splitlines() if x.strip()]
                # DISCARD THE FIRST SAMPLE: neuron-monitor's initial report is always
                # 0% (it has no previous interval to diff against). Reading only
                # line[0] -- the obvious thing to do -- yields a confident 0.0%
                # even while both cores are pegged. Verified directly under load.
                for ln in lines[1:]:
                    d = json.loads(ln)
                    # Each Triton instance is its OWN Neuron runtime pinned to ONE
                    # core, and each runtime reports 0 for the cores it does not own.
                    # So take the MAX per core across runtimes, then average over
                    # cores -- averaging every value would halve the result.
                    per_core = {}
                    for rt in d.get('neuron_runtime_data', []):
                        nc = rt['report']['neuroncore_counters']['neuroncores_in_use']
                        for k, v in nc.items():
                            u = v['neuroncore_utilization']
                            per_core[k] = max(per_core.get(k, 0.0), u)
                    if per_core:
                        self.dev.append(sum(per_core.values()) / len(per_core))
            except Exception:
                pass

    def __enter__(self):
        self._t = threading.Thread(target=self._run, daemon=True)
        self._t.start()
        return self

    def __exit__(self, *a):
        self._stop.set()
        if self._t:
            self._t.join(timeout=3)

    def summary(self):
        return (max(self.cpu) if self.cpu else None,
                (sum(self.cpu) / len(self.cpu)) if self.cpu else None,
                (sum(self.dev) / len(self.dev)) if self.dev else None)


def make_inputs(bs):
    e = tok(['What is machine learning?'] * bs,
            ['Machine learning is a subset of artificial intelligence that focuses on '
             'building systems that learn from data.'] * bs,
            max_length=SEQ_LENGTH, padding='max_length', truncation=True,
            return_tensors='np')
    return e['input_ids'].astype(np.int64), e['attention_mask'].astype(np.int64)

def _mk(ids_np, mask_np):
    a = httpclient.InferInput('input_ids', ids_np.shape, 'INT64')
    b = httpclient.InferInput('attention_mask', mask_np.shape, 'INT64')
    a.set_data_from_numpy(ids_np); b.set_data_from_numpy(mask_np)
    return [a, b]

def worker(client, bs, dur, q, stop):
    ids_np, mask_np = make_inputs(bs)
    inp = _mk(ids_np, mask_np)
    end = time.time() + dur
    while time.time() < end and not stop.is_set():
        try:
            t0 = time.time()
            client.infer(model_name=MODEL_NAME, inputs=inp)
            q.put((time.time() - t0) * 1000)
        except Exception:
            break

def run(bs, nworkers, dur=DURATION):
    clients = [httpclient.InferenceServerClient(url=TRITON_URL) for _ in range(nworkers)]
    ids_np, mask_np = make_inputs(bs)
    inp = _mk(ids_np, mask_np)
    for _ in range(5):                      # client-side warmup
        clients[0].infer(model_name=MODEL_NAME, inputs=inp)
    q, stop, threads = Queue(), threading.Event(), []
    with HostMon() as mon:
        t0 = time.time()
        for c in clients:
            t = threading.Thread(target=worker, args=(c, bs, dur, q, stop))
            t.start(); threads.append(t)
        for t in threads:
            t.join()
        elapsed = time.time() - t0
    cpu_max, cpu_avg, dev_avg = mon.summary()
    lat = []
    while not q.empty():
        lat.append(q.get())
    if not lat:
        return None
    return {'batch': bs, 'workers': nworkers, 'requests': len(lat),
            'p50': float(np.percentile(lat, 50)), 'p95': float(np.percentile(lat, 95)),
            'p99': float(np.percentile(lat, 99)),
            'throughput': len(lat) * bs / elapsed,
            'host_cpu_avg': cpu_avg, 'host_cpu_max': cpu_max,
            'device_util_avg': dev_avg, 'vcpu': NUM_VCPU}

rows = []
hdr = (f'{"BS":>3} {"W":>4} {"p50 ms":>9} {"p95 ms":>9} {"p99 ms":>9} '
       f'{"inf/s":>9} {"hostCPU%":>9} {"devUtil%":>9}')
print(hdr)
print('-' * len(hdr))
for nw in [1, 8, 16, 32]:
    for bs in BATCH_SIZES:
        r = run(bs, nw)
        if r:
            rows.append(r)
            ca = f'{r["host_cpu_avg"]:.0f}' if r['host_cpu_avg'] is not None else '-'
            dv = f'{r["device_util_avg"]:.0f}' if r['device_util_avg'] is not None else '-'
            print(f'{r["batch"]:>3} {r["workers"]:>4} {r["p50"]:>9.2f} '
                  f'{r["p95"]:>9.2f} {r["p99"]:>9.2f} {r["throughput"]:>9.1f} '
                  f'{ca:>9} {dv:>9}')

with open(os.path.expanduser('~/triton_bench_inf2_xlarge.json'), 'w') as f:
    json.dump(rows, f, indent=2)

best = max(rows, key=lambda r: r['throughput'])
print(f'\nPEAK: {best["throughput"]:.1f} inf/s at BS={best["batch"]}, '
      f'{best["workers"]} workers (p50 {best["p50"]:.2f} ms)')

# ---- bottleneck verdict, from the measurements rather than from assumption ----
cpus = [r['host_cpu_avg'] for r in rows if r['host_cpu_avg'] is not None]
devs = [r['device_util_avg'] for r in rows if r['device_util_avg'] is not None]
print('\n=== Bottleneck evidence ===')
if cpus:
    print(f'host CPU avg across configs : {sum(cpus)/len(cpus):.1f}%  '
          f'(peak {max(cpus):.1f}%) on {NUM_VCPU} vCPU')
if devs:
    print(f'device util avg            : {sum(devs)/len(devs):.1f}%')
if cpus and devs:
    c, d = max(cpus), max(devs)
    if c > 90 and d < 80:
        print('VERDICT: host-CPU-bound. The 4 vCPU host is the limit, not the device.')
    elif c < 80 and d > 90:
        print('VERDICT: device-bound. The smaller host costs nothing here.')
    elif c > 90 and d > 90:
        print('VERDICT: both host and device near saturation.')
    else:
        print('VERDICT: neither clearly saturated; likely latency/queueing-bound.')
print('\nNOTE: the benchmark client is co-resident on these same vCPU, so part of the '
      'host CPU figure is measurement overhead, not serving cost.')

host vCPU: 4


in-container neuron-monitor config: ok


neuron runtimes visible in container: 2
 BS    W    p50 ms    p95 ms    p99 ms     inf/s  hostCPU%  devUtil%
--------------------------------------------------------------------


  1    8     40.19     67.76     77.54     152.7        52        96


  2    8     80.52    133.64    137.53     149.4        54        98


...

  1   32    191.50    238.16    266.44     155.5        56        97


  2   32    409.81    433.37    438.11     149.1        55        98


  4   32    845.80    863.51    867.36     149.1        56        97


  8   32   1991.04   2007.37   2009.04     126.6        55        74


 16   32   3637.71   3881.79   3894.41     130.4        58        97

PEAK: 155.7 inf/s at BS=1, 16 workers (p50 102.01 ms)

=== Bottleneck evidence ===
host CPU avg across configs : 54.3%  (peak 57.6%) on 4 vCPU
device util avg            : 94.9%
VERDICT: device-bound. The smaller host costs nothing here.

NOTE: the benchmark client is co-resident on these same vCPU, so part of the host CPU figure is measurement overhead, not serving cost.


---
## Step 7: Results summary and bottleneck analysis

This step turns the raw benchmark rows into the three numbers that actually drive a
deployment decision, and states which resource is the limit.

### Reference figures from validated runs of this notebook

Measured on inf2.xlarge, SDK 2.31 DLAMI `20260813`, 2 instances, seq_len=1024. The table below
is one complete cold run (us-west-2). **Four independent full runs across two instances and
two regions peaked at 156.7 / 156.5 / 156.1 / 154.8 inf/s** -- a ~1.2% spread, which sets the
noise floor for any comparison you make against these numbers.

| BS | Workers | p50 ms | inf/s | host CPU % | device util % |
|---|---|---|---|---|---|
| **1** | **8** | **39.76** | **152.8** | 52.7 | 95.1 |
| 2 | 8 | 106.56 | 148.8 | 54.5 | 96.8 |
| 4 | 8 | 208.94 | 150.4 | 54.6 | 97.5 |
| 8 | 8 | 468.19 | 125.8 | 53.4 | 98.9 |
| 16 | 8 | 747.47 | 130.1 | 52.7 | 96.4 |
| 1 | 16 | 102.56 | **154.8 (peak)** | 55.4 | 95.7 |
| 4 | 16 | 377.92 | 149.9 | 54.2 | 98.5 |
| 1 | 32 | 205.23 | 154.6 | 56.9 | 96.0 |
| 16 | 32 | 3604.39 | 131.0 | 55.8 | 96.4 |

Peak sits at BS=1 with high concurrency, but **which** high concurrency (16 vs 32) is noise:
154.8 vs 154.6 inf/s is a 0.1% difference.

Single-core, non-Triton reference on this model for context: **67.62 qps** at 236.63 ms
(BS=16, seq=1024, SDK 2.31 + eager). Triton with 2 instances and dynamic batching roughly
**2.3x** that.

### Reading the results

**Throughput peaks at BS=1 with high client concurrency**, not at large client-side batches.
Server-side dynamic batching is doing the batching work, which means **clients can stay
simple** -- send one pair at a time and let Triton assemble batches.

**Latency, not throughput, is where the configuration choice matters.** In the run above,
152.8 inf/s at **39.76 ms** p50 (BS=1, W=8) versus 154.8 inf/s at **102.56 ms** (BS=1, W=16):
the last **1.3%** of throughput costs **2.6x the latency**. Across runs this tradeoff ranged
from 2.6x to 3.8x, always for under ~2% throughput. For a reranker in a search path,
**BS=1 with 8 workers is the right operating point**; the cell below identifies it
automatically from your own measurements.

**Large client-side batches are counterproductive here.** BS=8 and BS=16 drop to ~126-131
inf/s while latency grows proportionally -- they add queueing delay without adding throughput.

### Cost

On-demand us-east-2: **inf2.xlarge is $0.7582/hr**, which works out to roughly
**202-206 inf/s per $/hr** and about **$1.38 per million inferences** at the recommended
operating point. The cell below computes both from your own measured numbers.


In [12]:
PRICE_PER_HR = 0.7582     # inf2.xlarge on-demand, us-east-2

peak_now = max(rows, key=lambda r: r['throughput'])
base_now = next((r for r in rows if r['batch'] == 1 and r['workers'] == 1), None)

print('=== Headline numbers ===')
print(f'peak throughput   : {peak_now["throughput"]:.1f} inf/s '
      f'(BS={peak_now["batch"]}, {peak_now["workers"]} workers, p50 {peak_now["p50"]:.2f} ms)')
if base_now:
    print(f'single-request    : {base_now["throughput"]:.1f} inf/s '
          f'at {base_now["p50"]:.2f} ms p50')

# Latency-sensitive operating point: best throughput under 100 ms p50.
cands = [r for r in rows if r['p50'] < 100]
if cands:
    b = max(cands, key=lambda r: r['throughput'])
    print(f'best < 100ms p50  : {b["throughput"]:.1f} inf/s at BS={b["batch"]}, '
          f'{b["workers"]} workers (p50 {b["p50"]:.2f} ms)')
    tradeoff = (peak_now['throughput'] / b['throughput'] - 1) * 100
    lat_cost = peak_now['p50'] / b['p50']
    print(f'  -> chasing peak buys {tradeoff:+.1f}% throughput for {lat_cost:.1f}x the p50.')
    if tradeoff < 5 and lat_cost > 2:
        print(f'  -> RECOMMENDED OPERATING POINT: BS={b["batch"]}, {b["workers"]} workers.')

print(f'\n=== Cost (inf2.xlarge, ${PRICE_PER_HR}/hr on-demand, us-east-2) ===')
print(f'peak inf/s per $/hr : {peak_now["throughput"] / PRICE_PER_HR:.1f}')
if cands:
    print(f'low-latency inf/s per $/hr : {b["throughput"] / PRICE_PER_HR:.1f}')
    per_m = PRICE_PER_HR / (b['throughput'] * 3600) * 1e6
    print(f'cost per 1M inferences     : ${per_m:.2f}  (at the low-latency point)')

print('\n=== Which resource is the limit? ===')
cpus = [r['host_cpu_avg'] for r in rows if r['host_cpu_avg'] is not None]
devs = [r['device_util_avg'] for r in rows if r['device_util_avg'] is not None]
if cpus:
    print(f'host CPU  : {sum(cpus)/len(cpus):.1f}% avg, {max(cpus):.1f}% peak '
          f'on {peak_now.get("vcpu", 4)} vCPU')
if devs:
    print(f'device    : {sum(devs)/len(devs):.1f}% avg, '
          f'{sum(1 for x in devs if x >= 96)}/{len(devs)} configs at >=96%')
if cpus and devs:
    c, d = max(cpus), max(devs)
    if c > 90 and d < 80:
        print('VERDICT: host-CPU-bound - the 4 vCPU host is the limit.')
    elif c < 80 and d > 90:
        print('VERDICT: DEVICE-BOUND. The 4 vCPU host is NOT the limit - it has '
              'headroom to spare while both NeuronCores are saturated.')
        print('  This is the expected result for this model, and it is why '
              'inf2.xlarge is the right size: a larger host would add cost '
              'without adding throughput.')
    elif c > 90 and d > 90:
        print('VERDICT: both host and device near saturation.')
    else:
        print('VERDICT: neither clearly saturated; likely latency/queueing-bound.')
print('\nNOTE: the benchmark client runs on these same vCPU, so part of the host CPU '
      'figure is measurement overhead rather than serving cost. A deployment with an '
      'off-box client has more headroom than shown.')

=== Headline numbers ===
peak throughput   : 155.7 inf/s (BS=1, 16 workers, p50 102.01 ms)
best < 100ms p50  : 152.7 inf/s at BS=1, 8 workers (p50 40.19 ms)
  -> chasing peak buys +1.9% throughput for 2.5x the p50.
  -> RECOMMENDED OPERATING POINT: BS=1, 8 workers.

=== Cost (inf2.xlarge, $0.7582/hr on-demand, us-east-2) ===
peak inf/s per $/hr : 205.3
low-latency inf/s per $/hr : 201.4
cost per 1M inferences     : $1.38  (at the low-latency point)

=== Which resource is the limit? ===
host CPU  : 54.3% avg, 57.6% peak on 4 vCPU
device    : 94.9% avg, 12/15 configs at >=96%
VERDICT: DEVICE-BOUND. The 4 vCPU host is NOT the limit - it has headroom to spare while both NeuronCores are saturated.
  This is the expected result for this model, and it is why inf2.xlarge is the right size: a larger host would add cost without adding throughput.

NOTE: the benchmark client runs on these same vCPU, so part of the host CPU figure is measurement overhead rather than serving cost. A deployment with

---
## Step 8: Sample reranking

End-to-end demonstration: one query, several passages, ranked by relevance.


In [13]:
client = httpclient.InferenceServerClient(url=TRITON_URL)

# Same query/passages the accuracy gate used, so the ranking here is directly comparable.
QUERY = 'What are the benefits of renewable energy?'
PASSAGES = [
    'Renewable energy sources like solar and wind power produce electricity without '
    'greenhouse gas emissions, reducing climate change impacts.',
    'The stock market experienced significant volatility in Q4, with tech stocks leading '
    'the decline amid rising interest rates.',
    'Solar panels have become 90% cheaper over the past decade, making renewable energy '
    'cost-competitive with fossil fuels.',
    'The history of ancient Rome spans over a thousand years, from its founding in 753 BC.',
    'Wind energy creates more jobs per megawatt than coal or natural gas power plants.',
    'Python is a popular programming language known for its readability.',
]
RELEVANT = {0, 2, 4}   # the renewable-energy passages

scored = []
for i, passage in enumerate(PASSAGES):
    e = tok([QUERY], [passage], max_length=SEQ_LENGTH, padding='max_length',
            truncation=True, return_tensors='np')
    inp = _mk(e['input_ids'].astype(np.int64), e['attention_mask'].astype(np.int64))
    res = client.infer(model_name=MODEL_NAME, inputs=inp)
    scored.append((float(res.as_numpy('score')[0][0]), i, passage))

scored.sort(key=lambda x: -x[0])

print(f'Query: "{QUERY}"\n')
print(f'{"Rank":>4} {"Score":>9} {"Prob":>7}  Passage')
print('-' * 92)
for rank, (score, idx, passage) in enumerate(scored, 1):
    prob = 1 / (1 + np.exp(-score))
    print(f'{rank:>4} {score:>+9.3f} {prob:>7.3f}  [{idx}] {passage[:62]}...')

top3 = {idx for _, idx, _ in scored[:3]}
print(f'\ntop-3 == relevant passages {sorted(RELEVANT)}: {top3 == RELEVANT}')

Query: "What are the benefits of renewable energy?"

Rank     Score    Prob  Passage
--------------------------------------------------------------------------------------------
   1    +1.240   0.776  [0] Renewable energy sources like solar and wind power produce ele...
   2    +0.920   0.715  [4] Wind energy creates more jobs per megawatt than coal or natura...
   3    +0.905   0.712  [2] Solar panels have become 90% cheaper over the past decade, mak...
   4    -3.307   0.035  [1] The stock market experienced significant volatility in Q4, wit...
   5    -3.321   0.035  [5] Python is a popular programming language known for its readabi...
   6    -3.327   0.035  [3] The history of ancient Rome spans over a thousand years, from ...

top-3 == relevant passages [0, 2, 4]: True


---
## Step 9: Cleanup

Stops and removes the container. Compiled graphs under `~/triton_repo` are kept so a rerun
skips compilation. **Terminate the instance** when finished.


In [14]:
sh(['docker', 'stop', CONTAINER])
sh(['docker', 'rm', CONTAINER])
out, _, _ = sh(['docker', 'ps', '-a', '--filter', f'name={CONTAINER}'])
print(out)
print('\nKept for reuse:')
print(f'  compiled graphs: ~/triton_repo/bert_reranker/1/*.pt')
print(f'  docker image   : {DOCKER_IMAGE}')
print(f'  benchmark data : ~/triton_bench_inf2_xlarge.json')
print('\nRemember to terminate the instance.')

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES

Kept for reuse:
  compiled graphs: ~/triton_repo/bert_reranker/1/*.pt
  docker image   : triton-neuron-bert-reranker:sdk231
  benchmark data : ~/triton_bench_inf2_xlarge.json

Remember to terminate the instance.
